# WorldSim v3.2 — CoT Field Reorder + Retrain
Move reasoning before action in 9 logic tasks, regenerate data, retrain 2B, compare

## 1. Environment & Schema Verification

In [ ]:
from pathlib import Path
import copy
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import UTC, datetime

import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

from scripts.common import read_jsonl, write_jsonl
from scripts.convert_mixed_final_to_training_format import convert_mixed_final_to_training_format
from scripts.curriculum_order_v3 import curriculum_order_v3
from scripts.generate_data import generate_dataset, load_batch_plan, load_generation_config
from scripts.validate_data import validate_file
from training.lib.output_schema import TASK_OUTPUT_SCHEMAS_V3

GGUF_DIR = REPO_ROOT / "artifacts" / "gguf"
LLAMA_SERVER = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-server"
LLAMA_QUANTIZE = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = REPO_ROOT / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"
CONFIG_DIR = REPO_ROOT / "config"
GENERATION = load_generation_config(CONFIG_DIR)
COT_TASKS = set(GENERATION.get("cot_reasoning_first_tasks", ["E", "F", "I", "J", "K", "L", "M", "R", "T"]))
V32_DATASET_ID = "worldsim-v32-cot-mix"
V32_TRAINING_DIR = REPO_ROOT / "data" / "training" / V32_DATASET_ID
V32_FINAL_DIR = REPO_ROOT / "data" / "final" / V32_DATASET_ID
V32_BATCH_ID = "batch_v32_cot_regen"
V32_BATCH_PATH = REPO_ROOT / "config" / "batches" / f"{V32_BATCH_ID}.yaml"
V32_FINAL_GGUF = GGUF_DIR / "worldsim-v32-cot-qwen3.5-2b-q4_k_m.gguf"
V31_FINAL_DIR = REPO_ROOT / "data" / "final" / "worldsim-v31-mix"
V31_GGUF = GGUF_DIR / "worldsim-v31-qwen3.5-2b-q4_k_m.gguf"

print(f"Schema release: {GENERATION.get('schema_release', 'unknown')}")
print(f"CoT tasks: {sorted(COT_TASKS)}")

choice_fields = {"action_id", "emotion", "priority_id", "coping_id", "social_action_id", "response_id", "decision_id", "action"}
reasoning_fields = {"personality_reasoning", "temperament_amplifier", "reasoning", "hint", "cause", "temperament_factor"}
for task_id in ["E", "F", "I", "J", "K", "L", "M", "R", "T"]:
    schema = TASK_OUTPUT_SCHEMAS_V3[task_id]
    fields = list(schema.model_fields.keys())
    first_choice_idx = next((idx for idx, field in enumerate(fields) if field in choice_fields), -1)
    first_reasoning_idx = next((idx for idx, field in enumerate(fields) if field in reasoning_fields), -1)
    ok = 0 <= first_reasoning_idx < first_choice_idx
    print(f"  Task {task_id}: {fields} -> reasoning@{first_reasoning_idx} before choice@{first_choice_idx}: {'OK' if ok else 'FAIL'}")


## 2. Regenerate CoT Tasks

In [ ]:
V31_COUNTS = {"E": 269, "F": 405, "I": 179, "J": 180, "K": 180, "L": 179, "M": 225, "R": 346, "T": 360}
TEACHER_MODEL = "google/gemini-3.1-flash-lite-preview"

batch_config = {
    "batch_id": V32_BATCH_ID,
    "schema_version": 3,
    "description": "v3.2 CoT regeneration for reasoning-first logic tasks",
    "tasks": V31_COUNTS,
}
V32_BATCH_PATH.parent.mkdir(parents=True, exist_ok=True)
with V32_BATCH_PATH.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(batch_config, handle, allow_unicode=True, sort_keys=False)

settings_override = copy.deepcopy(GENERATION)
settings_override.setdefault("provider", {})["default_model"] = TEACHER_MODEL
settings_override["provider"].setdefault("task_model_overrides", {})
for task_id in COT_TASKS:
    settings_override["provider"]["task_model_overrides"][task_id] = TEACHER_MODEL

batch_plan = load_batch_plan(REPO_ROOT, batch_config=V32_BATCH_PATH)
target = sum((batch_plan.get("task_counts") or batch_plan.get("tasks", {})).values())
print(f"Batch file: {V32_BATCH_PATH}")
print(f"Teacher model override: {TEACHER_MODEL}")
print(f"Target requests: {target}")

started = time.perf_counter()
generation_result = generate_dataset(
    REPO_ROOT,
    batch_plan=batch_plan,
    settings_override=settings_override,
)
elapsed = time.perf_counter() - started
print(f"Generation output: {generation_result.output_path}")
print(f"Completed rows: {generation_result.count}")
print(f"Skipped rows: {generation_result.skipped_count}")
print(f"Estimated cost: ${generation_result.estimated_cost_usd:.2f}")
print(f"Elapsed: {elapsed / 60:.1f} min")


## 3. Validate New CoT Data

In [ ]:
raw_output = generation_result.output_path
validated_dir = REPO_ROOT / "data" / "validated" / V32_BATCH_ID
validation_report = validate_file(input_path=raw_output, validated_dir=validated_dir, repo_root=REPO_ROOT)
new_cot_path = validated_dir / "passed.jsonl"
print(f"Validated dir: {validated_dir}")
print(f"Passed: {validation_report.get('passed', 0)}")
print(f"Failed: {validation_report.get('failed', 0)}")
print(f"Passed file exists: {new_cot_path.exists()}")


## 4. Merge v3.2 Dataset

In [ ]:
def stratified_split(rows, *, dev_ratio=0.1, seed=42):
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for row in rows:
        buckets[str(row.get("task", "unknown"))].append(row)
    train_rows, dev_rows = [], []
    for task_id in sorted(buckets):
        task_rows = list(buckets[task_id])
        rng.shuffle(task_rows)
        if len(task_rows) <= 1:
            train_rows.extend(task_rows)
            continue
        dev_count = max(1, int(len(task_rows) * dev_ratio))
        if dev_count >= len(task_rows):
            dev_count = len(task_rows) - 1
        dev_rows.extend(task_rows[:dev_count])
        train_rows.extend(task_rows[dev_count:])
    return train_rows, dev_rows

v31_rows = read_jsonl(V31_FINAL_DIR / "train.jsonl") + read_jsonl(V31_FINAL_DIR / "dev.jsonl")
non_cot_rows = [row for row in v31_rows if row.get("task") not in COT_TASKS]
new_cot_rows = [row for row in read_jsonl(new_cot_path) if row.get("task") in COT_TASKS]

train_rows, dev_rows = stratified_split(non_cot_rows + new_cot_rows, dev_ratio=0.1, seed=42)
V32_FINAL_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(V32_FINAL_DIR / "train.jsonl", train_rows)
write_jsonl(V32_FINAL_DIR / "dev.jsonl", dev_rows)

merge_manifest = {
    "dataset_id": V32_DATASET_ID,
    "generated_at": datetime.now(UTC).isoformat(),
    "source_dataset": "worldsim-v31-mix",
    "replaced_cot_tasks": sorted(COT_TASKS),
    "counts": {
        "v31_total": len(v31_rows),
        "v31_non_cot": len(non_cot_rows),
        "new_cot": len(new_cot_rows),
        "train": len(train_rows),
        "dev": len(dev_rows),
    },
    "task_counts": {
        "train": dict(sorted(Counter(row.get("task", "?") for row in train_rows).items())),
        "dev": dict(sorted(Counter(row.get("task", "?") for row in dev_rows).items())),
    },
}
merge_manifest_path = V32_FINAL_DIR / "merge_manifest.json"
merge_manifest_path.write_text(json.dumps(merge_manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Non-CoT rows from v3.1: {len(non_cot_rows)}")
print(f"New CoT rows: {len(new_cot_rows)}")
print(f"Train rows: {len(train_rows)}")
print(f"Dev rows: {len(dev_rows)}")
print(f"Manifest: {merge_manifest_path}")


## 5. Convert To Training Format

In [ ]:
conversion_result = convert_mixed_final_to_training_format(
    repo_root=REPO_ROOT,
    input_train=V32_FINAL_DIR / "train.jsonl",
    input_dev=V32_FINAL_DIR / "dev.jsonl",
    source_manifest=merge_manifest_path,
    output_dir=V32_TRAINING_DIR,
    dataset_id=V32_DATASET_ID,
)

train_converted = conversion_result.train_output
dev_converted = conversion_result.dev_output
train_curriculum = V32_TRAINING_DIR / "train_curriculum.jsonl"
converted_rows = read_jsonl(train_converted)
ordered_rows = curriculum_order_v3(converted_rows, seed=42)
write_jsonl(train_curriculum, ordered_rows)

train_count = len(ordered_rows)
dev_count = conversion_result.dev_count
effective_batch = 1 * 8
max_steps = max(1, math.ceil(train_count / effective_batch)) * 3
l3_en_count = sum(1 for row in ordered_rows if row.get("messages", [{}])[0].get("content", "").startswith("You are WorldSim"))

print(f"Train: {train_count}, Dev: {dev_count}, Max steps: {max_steps}")
print(f"Rows using L3_EN system prompt: {l3_en_count}")


## 6. Train 2B v3.2 CoT Model

In [ ]:
from training.lib.qlora_smoke import SmokeRunConfig, run_baseline_or_raise

run_id = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%SZ")
training_output_dir = REPO_ROOT / "outputs" / "baseline" / "worldsim-v32-cot-2b" / run_id
cfg = SmokeRunConfig(
    run_mode="baseline",
    model_name="Qwen/Qwen3.5-2B-Base",
    train_file=train_curriculum,
    dev_file=dev_converted,
    output_dir=training_output_dir,
    max_steps=max_steps,
    max_train_samples=0,
    max_eval_samples=0,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_steps=100,
    save_steps=100,
    save_total_limit=1,
    require_qlora=True,
    seed=42,
    dry_run=False,
)

started = time.perf_counter()
train_result = run_baseline_or_raise(cfg)
train_elapsed = time.perf_counter() - started

print(f"Output: {training_output_dir}")
print(f"train_loss: {train_result.train_loss}")
print(f"eval_loss: {train_result.eval_loss}")
print(f"elapsed_min: {train_elapsed / 60:.1f}")


## 7. GGUF Conversion

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

adapter_dir = Path(train_result.output_dir) / "adapter"
merged_dir = Path(train_result.output_dir) / "merged_bf16"
fp16_gguf = Path(train_result.output_dir) / "worldsim-v32-cot-2b-f16.gguf"
q4_gguf = Path(train_result.output_dir) / "worldsim-v32-cot-2b-q4_k_m.gguf"

if not (merged_dir / "config.json").exists():
    base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-2B-Base", torch_dtype=torch.bfloat16, device_map="cpu")
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B-Base")
    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
    merged_dir.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    del merged, base
    torch.cuda.empty_cache()

if not fp16_gguf.exists():
    proc = subprocess.run([sys.executable, str(CONVERT_SCRIPT), str(merged_dir), "--outfile", str(fp16_gguf), "--outtype", "f16"], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-300:])
if not q4_gguf.exists():
    proc = subprocess.run([str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), "Q4_K_M"], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-300:])
shutil.copy2(q4_gguf, V32_FINAL_GGUF)
print(f"Saved GGUF: {V32_FINAL_GGUF} ({V32_FINAL_GGUF.stat().st_size / 1e6:.0f} MB)")


## 8. Compare v3.1 SFT vs v3.2 CoT SFT

In [ ]:
import urllib.request

SERVER_PORT = 8090
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'


def start_server(model_path: Path, ctx_size: int = 2048, n_gpu_layers: int = 99):
    args = [
        str(LLAMA_SERVER), '-m', str(model_path),
        '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '-c', str(ctx_size), '-np', '1', '-ngl', str(n_gpu_layers),
        '-fa', 'on', '--jinja', '--no-webui',
        '--chat-template-kwargs', '{"enable_thinking": false}',
        '-ctk', 'q8_0', '-ctv', 'q8_0',
    ]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2)
            data = json.loads(resp.read())
            if data.get('status') == 'ok':
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            stderr = proc.stderr.read().decode(errors='ignore')[-300:]
            raise RuntimeError(f'Server died: {stderr}')
    proc.kill()
    raise RuntimeError('Server startup timeout')


def stop_server(proc) -> None:
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
            proc.wait()


def strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def query_model(messages, response_format=None, max_tokens: int = 256, temperature: float = 0):
    payload = {
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stream': False,
    }
    if response_format:
        payload['response_format'] = response_format

    req = urllib.request.Request(
        f'{SERVER_URL}/v1/chat/completions',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read())
        content = data['choices'][0]['message']['content']
        return strip_think(content)
    except Exception as exc:
        return f'ERROR: {exc}'

rng = random.Random(42)

SYS_L3 = 'You are WorldSim logic assistant. Output JSON only. Follow [TEMP], [STRESS], [WORLD] context. Respect enum values and numeric ranges exactly.'
SYS_L4 = '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라.'
SYS_L5 = '너는 WorldSim 신탁 해석 도우미다. JSON으로만 답하라.'


def pick(items, n=1):
    return rng.sample(items, min(n, len(items)))


def temp_block(t):
    return f"NS={t['ns']} HA={t['ha']} RD={t['rd']} P={t['p']} type={t['id']}"


def personality_keywords(p):
    return ', '.join(p.get('keywords', []))


def json_schema(name: str, required: list[str], properties: dict) -> dict:
    return {
        'type': 'json_schema',
        'json_schema': {
            'name': name,
            'strict': True,
            'schema': {
                'type': 'object',
                'additionalProperties': False,
                'required': required,
                'properties': properties,
            },
        },
    }


emotion_ids = [e['id'] for e in emotions]
speaker_roles = ['elder', 'hunter', 'shaman', 'warrior', 'healer', 'gatherer', 'craftsman', 'chief', 'scout', 'observer']

b_schema = json_schema(
    'task_b',
    ['text_ko', 'text_en', 'register', 'emotion_expressed', 'intensity', 'mimetics', 'temperament_influence'],
    {
        'text_ko': {'type': 'string'},
        'text_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number'},
        'mimetics': {'type': 'array', 'items': {'type': 'string'}},
        'temperament_influence': {'type': 'string'},
    },
)
c_schema = json_schema(
    'task_c',
    ['speech_ko', 'speech_en', 'register', 'emotion_expressed', 'speaker_role', 'temperament_tone'],
    {
        'speech_ko': {'type': 'string'},
        'speech_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'speaker_role': {'type': 'string', 'enum': speaker_roles},
        'temperament_tone': {'type': 'string'},
    },
)
e_schema_for = lambda option_count: json_schema(
    'task_e',
    ['action_id', 'confidence', 'hint', 'personality_reasoning', 'temperament_factor'],
    {
        'action_id': {'type': 'integer', 'enum': list(range(option_count))},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'hint': {'type': 'string'},
        'personality_reasoning': {'type': 'string', 'enum': ['novelty_seeking', 'harm_avoidance', 'reward_dependence', 'persistence']},
        'temperament_factor': {'type': 'string'},
    },
)
f_schema = json_schema(
    'task_f',
    ['emotion', 'intensity', 'cause', 'previous_emotion', 'transition_type', 'temperament_amplifier'],
    {
        'emotion': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'cause': {'type': 'string'},
        'previous_emotion': {'type': 'string', 'enum': emotion_ids},
        'transition_type': {'type': 'string', 'enum': ['gradual', 'sudden', 'sustained']},
        'temperament_amplifier': {'type': 'string'},
    },
)
g_schema = json_schema(
    'task_g',
    ['interpretation_ko', 'interpretation_en', 'action_tendency', 'confidence', 'register', 'misinterpretation_type', 'temperament_bias'],
    {
        'interpretation_ko': {'type': 'string'},
        'interpretation_en': {'type': 'string'},
        'action_tendency': {'type': 'string', 'enum': ['mobilize', 'defend', 'wait', 'retreat', 'celebrate', 'mourn']},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'register': {'type': 'string', 'enum': ['haera', 'hao', 'hae']},
        'misinterpretation_type': {'type': 'string', 'enum': ['overconfident_literal', 'cautious_reversal', 'optimistic_expansion', 'passive_deferral', 'symbolic_abstraction']},
        'temperament_bias': {'type': 'string'},
    },
)
m_schema = json_schema(
    'task_m',
    ['decision_id', 'confidence', 'dissent_risk', 'reasoning', 'resource_commitment', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'reasoning': {'type': 'string'},
        'resource_commitment': {'type': 'string', 'enum': ['food', 'tools', 'labor', 'weapons', 'none']},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)
o_schema = json_schema(
    'task_o',
    ['public_claim', 'private_truth', 'deception_style', 'lie_degree', 'detection_risk', 'confidence'],
    {
        'public_claim': {'type': 'string'},
        'private_truth': {'type': 'string'},
        'deception_style': {'type': 'string', 'enum': ['evasion', 'half_truth', 'outright_lie', 'exaggeration', 'omission']},
        'lie_degree': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'detection_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
p_schema = json_schema(
    'task_p',
    ['retold_version', 'distortion_type', 'added_detail', 'dropped_detail', 'emotional_charge'],
    {
        'retold_version': {'type': 'string'},
        'distortion_type': {'type': 'string', 'enum': ['exaggeration', 'minimization', 'malicious_twist', 'emotional_coloring', 'detail_invention', 'source_confusion', 'faithful']},
        'added_detail': {'type': 'string'},
        'dropped_detail': {'type': 'string'},
        'emotional_charge': {'type': 'number', 'minimum': -1, 'maximum': 1},
    },
)
q_schema = json_schema(
    'task_q',
    ['trauma_response', 'behavioral_change', 'trigger_situation', 'intensity', 'duration', 'coping_mechanism'],
    {
        'trauma_response': {'type': 'string', 'enum': ['avoidance', 'overprotection', 'aggression', 'withdrawal', 'hypervigilance', 'ritual_coping', 'resilience']},
        'behavioral_change': {'type': 'string'},
        'trigger_situation': {'type': 'string'},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'duration': {'type': 'string', 'enum': ['short_term', 'long_term', 'permanent']},
        'coping_mechanism': {'type': 'string'},
    },
)
r_schema = json_schema(
    'task_r',
    ['action', 'counter_give', 'counter_want', 'reasoning', 'emotional_state', 'walk_away_threshold'],
    {
        'action': {'type': 'string', 'enum': ['accept', 'counter_offer', 'reject', 'walk_away', 'stall', 'bluff']},
        'counter_give': {'type': 'string'},
        'counter_want': {'type': 'string'},
        'reasoning': {'type': 'string'},
        'emotional_state': {'type': 'string', 'enum': emotion_ids},
        'walk_away_threshold': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
t_schema = json_schema(
    'task_t',
    ['decision_id', 'confidence', 'dissent_risk', 'minority_position', 'minority_action', 'spark_event', 'reasoning', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'minority_position': {'type': 'integer'},
        'minority_action': {'type': 'string', 'enum': ['comply', 'grumble', 'passive_resist', 'splinter', 'coup_attempt']},
        'spark_event': {'type': 'string', 'enum': ['food_shortage', 'battle_loss', 'oracle_conflict', 'leader_death', 'betrayal', 'resource_discovery']},
        'reasoning': {'type': 'string'},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)

ALL_PROMPTS = []
pair_counter = 0

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'B',
            'pair_id': f"B-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"B: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] B\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[기질 키워드] {temp['keywords']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[WORLD] default: 석기시대\n'
                '[출력 형식]\n'
                '{"text_ko": "해라체 2-3문장", "text_en": "English", "register": "haera", "emotion_expressed": "emotion", "intensity": 0.0, "mimetics": [], "temperament_influence": "str"}'
            ),
            'response_format': b_schema,
        })

for sit in pick(situations, 5):
    options = sit.get('action_options', sit.get('typical_actions', []))
    if not options:
        continue
    options_line = ' '.join(f"{i}:{o}" for i, o in enumerate(options))
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'E',
            'pair_id': f"E-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"E: {sit['id']} / {temp['id']}",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] E - Action Selection\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[EMOTION] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                f"[OPTIONS]\n{options_line}\n"
                '[OUTPUT FORMAT]\n'
                '{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'
            ),
            'response_format': e_schema_for(len(options)),
        })

for sit in pick(situations, 5):
    prev_emotion = rng.choice(emotions)
    for temp in [TEMPERAMENTS[2], TEMPERAMENTS[3]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'F',
            'pair_id': f"F-{len(ALL_PROMPTS) // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"F: {sit['id']} / {temp['id']} (prev={prev_emotion['id']})",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] F - Emotion Judgment\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[CURRENT EMOTION] {prev_emotion['id']}:{rng.choice(prev_emotion.get('intensities', [0.5]))}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                '[OUTPUT FORMAT]\n'
                f'{{"emotion": "one of 8", "intensity": 0.0-1.0, "cause": "sentence", "previous_emotion": "{prev_emotion["id"]}", "transition_type": "gradual|sudden|sustained", "temperament_amplifier": "str"}}'
            ),
            'response_format': f_schema,
        })

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    register = pers_register = 'haera'
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'C',
            'pair_id': None,
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"C: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] C\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.6]))}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[역할] warrior\n'
                '[출력 형식]\n'
                '{"speech_ko": "해라체 대사", "speech_en": "English", "register": "haera", "emotion_expressed": "emotion", "speaker_role": "role", "temperament_tone": "str"}'
            ),
            'response_format': c_schema,
        })

for sit in pick(situations, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'G',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'oracle',
        'scenario_id': sit['id'],
        'desc': f"G: {sit['id']} / {temp['id']}",
        'system': SYS_L5,
        'user_prompt': (
            f"[TASK] G - Oracle Interpretation\n[TEMP]\n{temp_block(temp)}\n"
            f"[기질 이름] {temp['id']}\n"
            f"[인물 성격] {temp['keywords']}\n"
            '[ORACLE]\n"큰물이 밀려오기 전에 높은 곳으로 가라"\n'
            f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
            '[역할] warrior\n'
            '[출력 형식]\n'
            '{"interpretation_ko": "해라체", "interpretation_en": "English", "action_tendency": "enum", "confidence": 0-1, "register": "haera", "misinterpretation_type": "enum", "temperament_bias": "str"}'
        ),
        'response_format': g_schema,
    })

for scenario in pick(deception_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    emotion = rng.choice(emotions)
    ALL_PROMPTS.append({
        'task': 'O',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"O: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] O - Deception\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.5]))}\n"
            f"[TRUE_STATE] {scenario['true_state']}\n"
            f"[PUBLIC_GOAL] {scenario['public_goal']}\n"
            f"[TARGET] {scenario['target']}\n"
            f"[DETECTION_CONTEXT] {scenario['detection_context']}\n"
            '[OUTPUT FORMAT]\n'
            '{"public_claim": "str", "private_truth": "str", "deception_style": "enum", "lie_degree": 0-1, "detection_risk": 0-1, "confidence": 0-1}'
        ),
        'response_format': o_schema,
    })

for scenario in pick(rumor_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'P',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"P: {scenario['id']} / {temp['id']} / {scenario['reteller_relationship']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] P - Rumor\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {rng.choice(emotions)['id']}:{round(rng.uniform(0.3, 0.8), 1)}\n"
            f"[ORIGINAL_FACT] {scenario['original_fact']}\n"
            f"[RETELLER_RELATIONSHIP] {scenario['reteller_relationship']}\n"
            '[OUTPUT FORMAT]\n'
            '{"retold_version": "str", "distortion_type": "enum", "added_detail": "str", "dropped_detail": "str", "emotional_charge": -1 to 1}'
        ),
        'response_format': p_schema,
    })

for scenario in pick(trauma_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'Q',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"Q: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] Q - Trauma\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] sadness:{round(rng.uniform(0.6, 0.95), 2)}\n"
            f"[TRAUMA_EVENT] {scenario['event']}\n"
            f"[TIME_SINCE] {scenario['time_since']}\n"
            f"[CURRENT_SITUATION] {scenario['current_situation']}\n"
            '[OUTPUT FORMAT]\n'
            '{"trauma_response": "enum", "behavioral_change": "str", "trigger_situation": "str", "intensity": 0-1, "duration": "enum", "coping_mechanism": "str"}'
        ),
        'response_format': q_schema,
    })

for scenario in pick(negotiation_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'R',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"R: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] R - Negotiate\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] anticipation:{round(rng.uniform(0.3, 0.7), 1)}\n"
            f"[CONTEXT] {scenario['context']}\n"
            f"[OUR_RESOURCES] {scenario['our_resources']}\n"
            f"[THEIR_RESOURCES] {scenario['their_resources']}\n"
            f"[OFFER_HISTORY] {scenario['offer_history']}\n"
            f"[POWER_BALANCE] {scenario['power_balance']}\n"
            '[OPTIONS]\n0:accept 1:counter_offer 2:reject 3:walk_away 4:stall 5:bluff\n'
            '[OUTPUT FORMAT]\n'
            '{"action": "enum", "counter_give": "str", "counter_want": "str", "reasoning": "str", "emotional_state": "emotion", "walk_away_threshold": 0-1}'
        ),
        'response_format': r_schema,
    })

for scenario in pick(group_situations, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o.get('ko', o.get('desc', ''))}" for o in opts) if opts else '0:agree 1:disagree 2:delay'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'M',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"M: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] M - Group Decision\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.3 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', scenario.get('desc', scenario['id']))}\n"
            f"[SITUATION] {scenario.get('situation', scenario.get('desc', scenario['id']))}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "reasoning": "str", "resource_commitment": "enum", "timeline": "enum"}'
        ),
        'response_format': m_schema,
    })

for scenario in pick(group_dissent_scenarios, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o['desc']}" for o in opts) if opts else '0:exile 1:trial 2:forgive'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'T',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"T: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] T - Group Dissent\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.4 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', '')}\n"
            f"[SITUATION] {scenario.get('situation', scenario['id'])}\n"
            f"[FACTION_HINT] {scenario.get('faction_hint', '')}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "minority_position": number, "minority_action": "enum", "spark_event": "enum", "reasoning": "str", "timeline": "enum"}'
        ),
        'response_format': t_schema,
    })

task_counts = Counter(prompt['task'] for prompt in ALL_PROMPTS)
print('=== Test Prompt Summary ===')
for task, count in sorted(task_counts.items()):
    paired = sum(1 for prompt in ALL_PROMPTS if prompt['task'] == task and prompt.get('pair_id'))
    print(f"  Task {task}: {count} prompts ({paired} paired)")
print(f"  Total: {len(ALL_PROMPTS)} prompts × {len(MODELS)} models = {len(ALL_PROMPTS) * len(MODELS)} inferences")


MODELS = {
    'v31_2b': {'path': V31_GGUF, 'label': '2B SFT v3.1'},
    'v32_2b': {'path': V32_FINAL_GGUF, 'label': '2B SFT v3.2 CoT'},
}

assert V31_GGUF.exists(), f"Missing v3.1 GGUF: {V31_GGUF}"
assert V32_FINAL_GGUF.exists(), f"Missing v3.2 GGUF: {V32_FINAL_GGUF}"

situations = yaml.safe_load((CONFIG_DIR / 'situations.yaml').read_text(encoding='utf-8'))['situations']
personalities = yaml.safe_load((CONFIG_DIR / 'personalities.yaml').read_text(encoding='utf-8'))['personalities']
emotions = yaml.safe_load((CONFIG_DIR / 'emotions.yaml').read_text(encoding='utf-8'))['emotions']
social_situations = yaml.safe_load((CONFIG_DIR / 'social_situations.yaml').read_text(encoding='utf-8'))['social_situations']
group_situations = yaml.safe_load((CONFIG_DIR / 'group_situations.yaml').read_text(encoding='utf-8'))['group_situations']
trade_scenarios = yaml.safe_load((CONFIG_DIR / 'trade_scenarios.yaml').read_text(encoding='utf-8'))['trade_scenarios']
stress_sources = yaml.safe_load((CONFIG_DIR / 'stress_sources.yaml').read_text(encoding='utf-8'))['stress_situations']
deception_scenarios = yaml.safe_load((CONFIG_DIR / 'deception_scenarios.yaml').read_text(encoding='utf-8'))['deception_scenarios']
rumor_scenarios = yaml.safe_load((CONFIG_DIR / 'rumor_scenarios.yaml').read_text(encoding='utf-8'))['rumor_scenarios']
trauma_scenarios = yaml.safe_load((CONFIG_DIR / 'trauma_scenarios.yaml').read_text(encoding='utf-8'))['trauma_scenarios']
negotiation_scenarios = yaml.safe_load((CONFIG_DIR / 'negotiation_scenarios.yaml').read_text(encoding='utf-8'))['negotiation_scenarios']
culture_scenarios = yaml.safe_load((CONFIG_DIR / 'culture_scenarios.yaml').read_text(encoding='utf-8'))['culture_scenarios']
group_dissent_scenarios = yaml.safe_load((CONFIG_DIR / 'group_dissent_scenarios.yaml').read_text(encoding='utf-8'))['group_dissent_scenarios']
TEMPERAMENTS = [
    {'id': 'choleric', 'ns': 0.8, 'ha': 0.2, 'rd': 0.5, 'p': 0.7, 'keywords': 'bold, impulsive, dominant'},
    {'id': 'melancholic', 'ns': 0.2, 'ha': 0.8, 'rd': 0.6, 'p': 0.4, 'keywords': 'cautious, anxious, detail-oriented'},
    {'id': 'sanguine', 'ns': 0.6, 'ha': 0.3, 'rd': 0.8, 'p': 0.3, 'keywords': 'sociable, warm, optimistic'},
    {'id': 'phlegmatic', 'ns': 0.3, 'ha': 0.5, 'rd': 0.4, 'p': 0.8, 'keywords': 'calm, patient, persistent'},
]

print('=' * 90)
print('  PERSONALITY CONSISTENCY — Paired Tests')
print('=' * 90)

pairs = defaultdict(list)
for row in RESULTS:
    pair_id = row.get('pair_id')
    if pair_id:
        pairs[(pair_id, row['model'])].append(row)

consistency_by_model = defaultdict(lambda: {'same': 0, 'different': 0, 'total': 0})

for (pair_id, model_key), pair_rows in sorted(pairs.items()):
    if len(pair_rows) != 2:
        continue
    first, second = pair_rows
    if not first['parsed'] or not second['parsed']:
        continue
    task = first['task']
    different = False
    if task == 'E':
        different = first['parsed'].get('action_id') != second['parsed'].get('action_id')
    elif task == 'B':
        different = (
            first['parsed'].get('emotion_expressed') != second['parsed'].get('emotion_expressed')
            or abs(first['parsed'].get('intensity', 0) - second['parsed'].get('intensity', 0)) > 0.1
        )
    elif task == 'F':
        different = first['parsed'].get('emotion') != second['parsed'].get('emotion')
    consistency_by_model[model_key]['total'] += 1
    if different:
        consistency_by_model[model_key]['different'] += 1
    else:
        consistency_by_model[model_key]['same'] += 1

model_order = ['base_08b', 'ft_08b', 'base_2b', 'ft_2b', 'base_4b', 'ft_4b', 'base_9b', 'ft_9b']
available = [model for model in model_order if model in MODELS]

print()
print(f"  {'Model':<20} {'Pairs':>6} {'Different':>10} {'Same':>6} {'Consistency%':>13}")
print(f"  {'-' * 58}")
for model_key in available:
    stats = consistency_by_model[model_key]
    total = stats['total']
    if total > 0:
        pct = stats['different'] / total * 100
        label = MODELS[model_key]['label']
        print(f"  {label:<20} {total:>6} {stats['different']:>10} {stats['same']:>6} {pct:>12.0f}%")

print()
print('  Higher % = model differentiates personalities better')
print('  Ideal: fine-tuned models should have higher consistency% than base')


RESULTS = []
for model_key, model_info in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  Testing: {model_info['label']}")
    print(f"{'=' * 60}")
    proc = start_server(model_info['path'])
    time.sleep(1)
    try:
        for index, test in enumerate(ALL_PROMPTS):
            messages = [
                {'role': 'system', 'content': test['system']},
                {'role': 'user', 'content': test['user_prompt']},
            ]
            started = time.perf_counter()
            output = query_model(messages, response_format=test.get('response_format'), max_tokens=256, temperature=0)
            latency = (time.perf_counter() - started) * 1000
            json_valid = False
            parsed = None
            try:
                parsed = json.loads(output)
                json_valid = True
            except Exception:
                pass
            RESULTS.append({
                'model': model_key,
                'model_label': model_info['label'],
                'task': test['task'],
                'desc': test.get('desc', ''),
                'pair_id': test.get('pair_id'),
                'temperament_id': test.get('temperament_id'),
                'output': output,
                'parsed': parsed,
                'json_valid': json_valid,
                'latency_ms': round(latency),
            })
            if (index + 1) % 20 == 0:
                print(f"  [{index + 1}/{len(ALL_PROMPTS)}] model={model_key}", flush=True)
    finally:
        stop_server(proc)
        time.sleep(2)


def auto_grade(result):
    grades = {}
    grades['json_valid'] = result['json_valid']
    if result['parsed']:
        values = [str(value) for value in result['parsed'].values() if isinstance(value, str)]
        placeholder_words = {'str', 'string', 'sentence', 'English', 'enum', 'number', '해라체', 'one of'}
        grades['not_placeholder'] = not any(value in placeholder_words for value in values)
    else:
        grades['not_placeholder'] = False
    if result['parsed']:
        str_fields = [value for value in result['parsed'].values() if isinstance(value, str)]
        avg_len = sum(len(value) for value in str_fields) / max(len(str_fields), 1)
        grades['text_richness'] = avg_len > 15
    else:
        grades['text_richness'] = False
    if result['parsed']:
        for key in ['confidence', 'intensity', 'lie_degree', 'detection_risk', 'dissent_risk', 'walk_away_threshold', 'emotional_charge']:
            if key in result['parsed']:
                value = result['parsed'][key]
                if isinstance(value, (int, float)):
                    grades['numeric_valid'] = (-1 <= value <= 1) if key == 'emotional_charge' else (0 <= value <= 1)
                    break
        if 'numeric_valid' not in grades:
            grades['numeric_valid'] = True
    else:
        grades['numeric_valid'] = False
    grades['score'] = sum([grades.get('json_valid', False), grades.get('not_placeholder', False), grades.get('text_richness', False), grades.get('numeric_valid', False)])
    return grades

for row in RESULTS:
    row['grades'] = auto_grade(row)

print('=' * 80)
print('  v3.1 SFT vs v3.2 CoT SFT')
print('=' * 80)
for model_key in ['v31_2b', 'v32_2b']:
    model_rows = [row for row in RESULTS if row['model'] == model_key]
    avg_score = sum(row['grades']['score'] for row in model_rows) / len(model_rows)
    json_pct = sum(1 for row in model_rows if row['grades']['json_valid']) / len(model_rows) * 100
    avg_ms = sum(row['latency_ms'] for row in model_rows) / len(model_rows)
    print(f"\n  {MODELS[model_key]['label']}: avg={avg_score:.2f}/4 json={json_pct:.0f}% latency={avg_ms:.0f}ms")

print('\nPer-task delta (v3.1 -> v3.2 CoT):')
for task in sorted(set(row['task'] for row in RESULTS)):
    v31_rows = [row for row in RESULTS if row['task'] == task and row['model'] == 'v31_2b']
    v32_rows = [row for row in RESULTS if row['task'] == task and row['model'] == 'v32_2b']
    if v31_rows and v32_rows:
        v31_avg = sum(row['grades']['score'] for row in v31_rows) / len(v31_rows)
        v32_avg = sum(row['grades']['score'] for row in v32_rows) / len(v32_rows)
        delta = v32_avg - v31_avg
        marker = '⭐' if delta > 0.3 else '✅' if delta > 0 else '⚠️' if delta == 0 else '❌'
        print(f"  {marker} Task {task}: {v31_avg:.2f} -> {v32_avg:.2f} (Δ={delta:+.2f})")


## 9. Personality Consistency Comparison

In [ ]:
pairs = defaultdict(list)
for row in RESULTS:
    pair_id = row.get('pair_id')
    if pair_id:
        pairs[(pair_id, row['model'])].append(row)

consistency = defaultdict(lambda: {'same': 0, 'different': 0, 'total': 0})
for (_, model_key), pair_rows in sorted(pairs.items()):
    if len(pair_rows) != 2:
        continue
    left, right = pair_rows
    if not left['parsed'] or not right['parsed']:
        continue
    task = left['task']
    different = False
    if task == 'E':
        different = left['parsed'].get('action_id') != right['parsed'].get('action_id')
    elif task == 'B':
        different = left['parsed'].get('emotion_expressed') != right['parsed'].get('emotion_expressed') or abs(left['parsed'].get('intensity', 0) - right['parsed'].get('intensity', 0)) > 0.1
    elif task == 'F':
        different = left['parsed'].get('emotion') != right['parsed'].get('emotion')
    consistency[model_key]['total'] += 1
    if different:
        consistency[model_key]['different'] += 1
    else:
        consistency[model_key]['same'] += 1

print(f"{'Model':<24} {'Pairs':>6} {'Different':>10} {'Same':>6} {'Consistency%':>13}")
print('-' * 64)
for model_key in ['v31_2b', 'v32_2b']:
    stats = consistency[model_key]
    total = stats['total']
    if total:
        pct = stats['different'] / total * 100
        print(f"{MODELS[model_key]['label']:<24} {total:>6} {stats['different']:>10} {stats['same']:>6} {pct:>12.0f}%")


## 10. Save Results

In [ ]:
results_path = REPO_ROOT / 'outputs' / 'benchmarks' / 'v32_cot_retrain_comparison.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
save_data = {
    'metadata': {
        'version': 'v3.2-cot',
        'schema_release': GENERATION.get('schema_release', 'v3.2-cot'),
        'cot_tasks': sorted(COT_TASKS),
        'generation_output_path': str(generation_result.output_path),
        'validated_output_path': str(new_cot_path),
        'train_count': train_count,
        'dev_count': dev_count,
        'training_loss': train_result.train_loss,
        'eval_loss': train_result.eval_loss,
        'training_time_min': round(train_elapsed / 60, 1),
        'total_results': len(RESULTS),
    },
    'results': [{key: value for key, value in row.items() if key != 'parsed'} for row in RESULTS],
}
with results_path.open('w', encoding='utf-8') as handle:
    json.dump(save_data, handle, indent=2, ensure_ascii=False)
print(f"Saved: {results_path}")
print(f"DPO/CoT train loss: {train_result.train_loss:.4f}")
print(f"Eval loss: {train_result.eval_loss:.4f}")
print(f"GGUF: {V32_FINAL_GGUF.name}")
